# Segunda codificação

## O que o estudo mede

Plataformas online testam coisas nos seus usuários todos os dias. Elas mudam o que
aparece no topo da lista, o texto de uma notificação, a posição de um botão, e medem
o efeito comparando grupos de pessoas. Isso se chama experimentação comportamental e
é rotina da indústria.

A pergunta do estudo é estreita e documental: **o que cada plataforma conta ao
usuário sobre isso, e em que tipo de documento ela conta?** Ninguém aqui vai
descobrir quais experimentos existem. O trabalho é ler o que a plataforma declara e
classificar a declaração.

O tipo de documento é o centro do estudo. Uma política de privacidade obriga a
plataforma perante o usuário. Um post de blog de engenharia não obriga nada. Quando
os dois falam do mesmo assunto em termos diferentes, essa diferença é o dado.

## Os 26 serviços

Não é uma amostra, é um censo. São os 26 serviços que a Comissão Europeia designou
sob o Digital Services Act: 24 como VLOP (*very large online platform*) e 2 como
VLOSE (*very large online search engine*). A designação depende de escala, acima de
45 milhões de usuários por mês na União Europeia, e traz obrigações próprias.

A unidade de análise é o serviço, não a empresa. O Google Ireland responde por
cinco deles (Search, Maps, Play, Shopping, YouTube) e a Meta por dois (Facebook,
Instagram). Quatro são plataformas de conteúdo adulto, e estão no censo porque são
VLOPs de pleno direito.

Cada serviço tem de 3 a 10 documentos, 157 no total: política de privacidade, termos
de uso, aviso de cookies, tabela de bases legais, e o que a plataforma publicou em
blog ou central de ajuda sobre o assunto.

## Por que existe uma segunda codificação

A primeira passada já foi feita, de forma automatizada. Uma classificação com fonte
única não separa duas coisas: quanto do resultado vem do texto e quanto vem de quem
leu o texto. Duas leituras independentes do mesmo material permitem medir a
concordância entre elas, e é essa medida que torna o resultado defensável diante de
um revisor.

A palavra independente tem uma consequência prática. **Este notebook não mostra o
que a primeira passada codificou, nem os resultados do estudo.** A razão não é
desconfiança. Se você soubesse o que se espera encontrar, sua leitura deixaria de
ser uma segunda medição e passaria a confirmar a primeira.

## O que você faz

Há uma célula por serviço, e as células são independentes entre si. Dentro de uma
célula o trabalho é sequencial: a tela mostra **uma variável por vez**. Você
preenche os campos dela, clica em salvar, e a mesma célula passa para a variável
seguinte. São dez ao todo, as nove variáveis mais o log de palavras-chave, e uma
linha no alto do painel mostra quais já fecharam e em qual você está.

A ordem é fixa de propósito: você responde a V1 antes de a V2 aparecer, e nenhuma
variável fecha sem evidência preenchida. Rever o que já respondeu é permitido,
pular adiante não.

Pode parar no meio e fechar o Colab. Quando voltar ao mesmo serviço, o painel abre
na primeira variável que ainda falta, com as anteriores já preenchidas. O progresso
é o mesmo do instrumento em HTML, então dá para alternar entre os dois.

O critério completo de cada variável está na célula **O codebook, variável por
variável**, logo antes dos serviços. Vale ler uma vez antes de começar.

## As duas listas de evidência, nessa ordem

Primeiro vem a **busca por palavra-chave**: os 12 termos do protocolo procurados
literalmente no texto congelado, endereçados por variável e com o trecho em volta.
Não passa por modelo nenhum. Por ser busca de texto, ela dá sempre o mesmo resultado
e não deixa nada de fora. Alguns trechos chegam com aviso, porque o termo costuma
aparecer em outro sentido ("Code of Ethics" no rodapé não é revisão ética de
experimento). Quem descarta é você.

Depois vem o que **o modelo** localizou. Ele acrescenta o que a palavra-chave não
alcança: passagem que descreve experimentação sem usar nenhum dos 12 termos. Nessa
ordem ele só pode somar.

**Nenhuma das duas atribui código.** A primeira passada já foi automatizada; se um
modelo decidisse aqui também, a concordância mediria o modelo contra ele mesmo. A
evidência do modelo foi gerada numa rodada única e congelada, então todo codificador
vê a mesma tela, e **você não precisa de chave de API para nada**.

## Duas regras que afetam o resultado

**Leia sempre do corpus congelado, nunca da página ao vivo.** As plataformas
reescrevem as políticas sem avisar. Se os dois codificadores lerem versões
diferentes do mesmo documento, a discordância entre vocês fica indistinguível de
mudança no documento, e depois não há como separar as duas.

**Confira o recibo embaixo do painel.** Cada variável que você fecha é gravada num
servidor e versionada, o que permite parar no meio, fechar o Colab e voltar depois,
inclusive de outra máquina. O recibo fica **verde** quando a resposta chegou ao
servidor e **vermelho** quando não chegou. Vermelho significa parar e avisar: o
arquivo local do Colab é apagado quando a sessão recicla, e o que estiver só nele se
perde.


In [ ]:
#@title Instalação e configuração { display-mode: "form" }
# Roda uma vez por sessão. ~15 s. O SDK da Anthropic não entra: a evidência do
# modelo já vem congelada, e este notebook não chama modelo nenhum.
!pip -q install ipywidgets

import hashlib, json, sys, urllib.request
from pathlib import Path

SITE = "https://experimented-without-consent-corpus.vercel.app"

# O runtime vem do mesmo host do corpus, não de um clone: os repositórios são
# privados, e pedir um token dentro de uma célula seria pior que o problema que
# resolve. Cada arquivo é conferido contra o manifesto — isso pega download
# truncado e cópia velha, não é barreira contra adversário.
destino = Path("/content/ewc")
manifesto = json.load(urllib.request.urlopen(f"{SITE}/lib/manifest.json", timeout=60))
for arquivo in manifesto["arquivos"]:
    with urllib.request.urlopen(f"{SITE}/lib/{arquivo['file']}", timeout=60) as r:
        dados = r.read()
    if hashlib.sha256(dados).hexdigest() != arquivo["sha256"]:
        raise SystemExit(f"{arquivo['file']}: download não bate com o manifesto")
    alvo = destino / arquivo["file"]
    alvo.parent.mkdir(parents=True, exist_ok=True)
    alvo.write_bytes(dados)
print(f"runtime publicado em {manifesto['publicado_em'][:10]} · "
      f"{len(manifesto['arquivos'])} arquivos conferidos")

sys.path.insert(0, str(destino / "analysis"))
import revisao as R

# O corpus é lido do mesmo site. Para trabalhar offline, baixe `md/` e aponte
# CORPUS para a pasta local — o resto do notebook não muda.
corpus = R.configurar(corpus=f"{SITE}/md", modelo="claude-opus-5")
print(f"{sum(len(s['docs']) for s in corpus.index['services'])} documentos · "
      f"{len(corpus.index['services'])} serviços")

# A procedência aparece UMA vez, aqui, e vale para as 26 células abaixo: a data do
# congelamento e o país da captura são do corpus inteiro, não de cada serviço.
# Cada painel leva só o carimbo de uma linha.
R.mostrar_procedencia(corpus)


## Sem chave de API

A evidência do modelo já está congelada e publicada ao lado do corpus, então o
notebook não chama modelo nenhum e não precisa de credencial.

A célula abaixo confirma que a evidência congelada chegou, e mostra a data em que
foi gerada. Se ela disser que não achou, me avise antes de começar a codificar:
sem ela o piso da busca por palavra-chave continua funcionando, mas você perde a
camada que o modelo acrescenta.


In [ ]:
# Confere a evidência congelada. Não chama modelo, não gasta nada.
import json, urllib.request
idx = json.load(urllib.request.urlopen(f"{SITE}/sugestoes/index.json", timeout=60))
total = sum(a["citacoes"] for a in idx["arquivos"])
print(f"evidência congelada: {len(idx['arquivos'])} serviços · {total} citações · "
      f"gerada em {idx['gerado_em'][:10]}")
print(f"modelo: {sorted({a['modelo'] for a in idx['arquivos']})}")


## O codebook, variável por variável

As dez variáveis com o critério completo: primeiro em linguagem direta, depois o
**texto exato do codebook congelado em 04/07/2026**, que é o que vale. Leia uma vez
antes de começar e volte aqui sempre que precisar.

Nas células de trabalho fica só o lembrete de uma linha, abaixo do título da
variável, e um ponteiro para cá. O critério inteiro repetido 26 vezes viraria
moldura. O que a célula de trabalho precisa ser é uma tela de decisão.

As âncoras do piloto **não** entram aqui. Elas nomeiam serviços, e num bloco único
não há serviço em tela contra o qual suprimi-las: mostrá-las poria a resposta de um
serviço no seu campo de visão enquanto você codifica outro.


In [ ]:
# Mostra as dez variáveis com o critério completo. Nada de rede, nada de modelo.
R.mostrar_codebook()


## Os 26 serviços

Cada célula abaixo é independente: rode na ordem que quiser, pare no meio,
volte depois. O que já foi respondido reaparece preenchido.


### Wikipedia


In [ ]:
R.painel("Wikipedia")


### Temu


In [ ]:
R.painel("Temu")


### Facebook


In [ ]:
R.painel("Facebook")


### YouTube


In [ ]:
R.painel("YouTube")


### Google Search


In [ ]:
R.painel("Google Search")


### WhatsApp


In [ ]:
R.painel("WhatsApp")


### Booking.com


In [ ]:
R.painel("Booking.com")


### Google Maps


In [ ]:
R.painel("Google Maps")


### Pornhub


In [ ]:
R.painel("Pornhub")


### Amazon Store


In [ ]:
R.painel("Amazon Store")


### LinkedIn


In [ ]:
R.painel("LinkedIn")


### Bing


In [ ]:
R.painel("Bing")


### XVideos


In [ ]:
R.painel("XVideos")


### Shein


In [ ]:
R.painel("Shein")


### X


In [ ]:
R.painel("X")


### XNXX


In [ ]:
R.painel("XNXX")


### Zalando


In [ ]:
R.painel("Zalando")


### Snapchat


In [ ]:
R.painel("Snapchat")


### Stripchat


In [ ]:
R.painel("Stripchat")


### App Store


In [ ]:
R.painel("App Store")


### Instagram


In [ ]:
R.painel("Instagram")


### Google Play


In [ ]:
R.painel("Google Play")


### Pinterest


In [ ]:
R.painel("Pinterest")


### Google Shopping


In [ ]:
R.painel("Google Shopping")


### TikTok


In [ ]:
R.painel("TikTok")


### AliExpress


In [ ]:
R.painel("AliExpress")


## Progresso


In [ ]:
# Onde a segunda passada está — metadados, sem revelar codificação.
e = R.F.Estado()
feitos = {s: sum(1 for k in r if not k.startswith("_") and r[k])
          for s, r in e.records.items()}
print(f"{sum(1 for n in feitos.values() if n >= 13)}/26 serviços com os 13 campos centrais")
for s, n in sorted(feitos.items(), key=lambda x: -x[1]):
    print(f"  {n:2d} campos · {s}")
